# BroadbandCare GRPO Training (Colab)

This notebook trains `Qwen2.5-1.5B-Instruct` with GRPO on the BroadbandCare tool environment.

In [ ]:
!pip install -q unsloth trl transformers accelerate bitsandbytes matplotlib pandas

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from broadbandcare.env.env import BroadbandSupportEnv
from broadbandcare.models import BroadbandcareAction

In [ ]:
from unsloth import FastLanguageModel

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

In [ ]:
env = BroadbandSupportEnv(seed=42)
obs = env.reset()
obs

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# NOTE: Replace `dataset` and `reward_fn` with your rollout pipeline.
config = GRPOConfig(
    output_dir="./grpo_broadbandcare",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    logging_steps=5,
    num_train_epochs=1,
    max_prompt_length=512,
    max_completion_length=128,
    beta=0.02,
)

print(config)

In [ ]:
# Example placeholder metrics. Replace this cell with real trainer logs.
metrics = {
    "epoch": [0, 1, 2, 3, 4],
    "loss": [2.4, 2.0, 1.7, 1.5, 1.4],
    "mean_episode_reward": [-1.2, -0.3, 0.8, 1.4, 2.2],
}
df = pd.DataFrame(metrics)
df

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(df["epoch"], df["loss"], marker="o")
ax[0].set_title("Training Loss")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")

ax[1].plot(df["epoch"], df["mean_episode_reward"], marker="o")
ax[1].set_title("Mean Episode Reward")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Reward")
plt.tight_layout()

In [ ]:
def run_eval_episode(env, tool_sequence):
    obs = env.reset()
    total_reward = 0.0
    for tool in tool_sequence:
        step_obs = env.step(BroadbandcareAction(tool=tool, args={}))
        total_reward += step_obs.reward
        if step_obs.done:
            break
    return {
        "total_reward": total_reward,
        "steps": step_obs.step_count,
        "done": step_obs.done,
        "metrics": step_obs.metadata,
    }

baseline = run_eval_episode(env, ["get_account_details", "resolve_ticket"])
candidate = run_eval_episode(env, ["get_account_details", "run_speed_test", "get_router_stats", "restart_router", "run_speed_test", "resolve_ticket"])
baseline, candidate